# Direct data access examples for the GGDC project

This notebook shows a series of direct data access methods, without downloading files locally. The examples are written from scratch and do not require installing the `geodata` library itself, only major Python libraries relevant to the example. Many other programming languages and libraries support the same underlying operations.

In [ ]:
## Data access handlers
from pystac_client import Client
import stackstac
import xarray as xr
import geopandas as gpd

## For ananymous authentication
import planetary_computer

## For plotting and basic computation examples
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches
import xdem
from rasterio.enums import Resampling

## Optional to silence a pyproj reprojection warning about loss of skew parameter
import warnings
warnings.filterwarnings('ignore')

## Copernicus DEM

### opentopgraphy VRT example

#### Load the global 30m product

In [ ]:
%%time
url = 'https://opentopography.s3.sdsc.edu/raster/COP30/COP30_hh.vrt'
ds = xr.open_dataarray(url,chunks='auto')
ds

#### Clip to Aletsch glacier bounds

In [ ]:
%%time
xmin, ymin, xmax, ymax = [7.92, 46.39, 8.14, 46.58]
clipped = ds.rio.clip_box(xmin, 
                          ymin, 
                          xmax, 
                          ymax).squeeze("band", 
                                        drop=True).rename("elevation").to_dataset()
clipped["elevation"]

#### Plot the DEM

In [ ]:
%%time
clipped['elevation'].plot()

#### Reproject and plot hillshade

In [ ]:
# reproject
clipped = clipped.rio.reproject("EPSG:2056",resampling=Resampling.bilinear)

In [ ]:
# compute hillshade
resolution = clipped.rio.resolution()[0]
hillshade = xdem.terrain.hillshade(clipped.elevation.values, resolution=resolution)
clipped['hillshade'] = (('y','x'), hillshade)

In [ ]:
# plot with equal aspect
fig,ax = plt.subplots(figsize = (7,7))
clipped['hillshade'].plot(ax=ax, cmap="Greys_r")
ax.set_aspect("equal")

#### Optionally save to disk

In [ ]:
# clipped['elevation'].rio.to_raster('clipped.tif')

### Microsoft Planetary Computer STAC example

In [ ]:
url = "https://planetarycomputer.microsoft.com/api/stac/v1"
collection = "cop-dem-glo-30"
asset_key = 'data'
bbox = [7.92, 46.39, 8.14, 46.58]
resolution = 30
epsg = 2056

catalog = Client.open(url, modifier=planetary_computer.sign_inplace)
search = catalog.search(collections=[collection], bbox=bbox, limit=50)
items = search.item_collection()

stack = stackstac.stack(items, 
                        assets=[asset_key], 
                        epsg=epsg, 
                        bounds_latlon=bbox,
                        resolution=resolution,
                        resampling=Resampling.bilinear,)

stack = stackstac.mosaic(stack, dim="time").squeeze(drop=True).rename("elevation").to_dataset()
stack['elevation']

In [ ]:
stack['elevation'].plot()

In [ ]:
# compute hillshade
resolution = stack.rio.resolution()[0]
hillshade = xdem.terrain.hillshade(stack.elevation.values, resolution=resolution)
stack['hillshade'] = (('y','x'), hillshade)

In [ ]:
# plot with equal aspect
fig,ax = plt.subplots(figsize = (7,7))
stack['hillshade'].plot(ax=ax, cmap="Greys_r")
ax.set_aspect("equal")

## ESA Worldcover

In [ ]:
url = 'https://planetarycomputer.microsoft.com/api/stac/v1'
collection = "esa-worldcover"
bbox = [7.92, 46.39, 8.14, 46.58]
epsg_code = 2056

catalog = Client.open(url, modifier=planetary_computer.sign_inplace)
search = catalog.search(collections=[collection],
                        bbox=bbox,
                       datetime="2021-01-01/2022-12-31",
                       )
items = search.item_collection()
stack = stackstac.stack(items, 
                        assets=["map"], 
                        epsg=epsg_code, 
                        bounds_latlon=bbox,
                        resampling=Resampling.bilinear)

stack = stack.squeeze("band", drop=True).rename("land_cover").to_dataset()
stack = stack.squeeze("time", drop=True)
stack["land_cover"]

### Reproject and plot

In [ ]:
fig,ax = plt.subplots(figsize = (7,7))

categories = items[0].assets["map"].extra_fields["classification:classes"]
values = [cat['value'] for cat in categories]
colors = ['#' + cat['color_hint'] for cat in categories]
labels = [cat['description'] for cat in categories]
cmap = ListedColormap(colors)
legend_handles = [mpatches.Patch(color=colors[i], label=f"{values[i]}: {labels[i]}") for i in range(len(values))]

stack["land_cover"].plot(ax=ax,cmap=cmap,add_colorbar=False)
ax.axes.set_title('ESA World Cover - Aletsch')
ax.axes.legend(handles=legend_handles, bbox_to_anchor=(1.05, 1), loc='upper left')
ax.set_aspect("equal")

## Global Administrative Areas vector data example
Direct read of a vector dataset

In [ ]:
zip_url = "https://geodata.ucdavis.edu/gadm/gadm4.1/shp/gadm41_CHE_shp.zip"
file_name = "gadm41_CHE_0.shp"

In [ ]:
vsi_path = f"/vsizip/vsicurl/{zip_url}/{file_name}"
gdf = gpd.read_file(vsi_path)

In [ ]:
gdf.explore()